In [ ]:
import pandas as pd
import numpy as np
from math import ceil
from io import BytesIO
from google.colab import files
from ipywidgets import FileUpload, IntText, Button, VBox, Output, Label
from IPython.display import display, clear_output

# Widgets de entrada
upload = FileUpload(accept='.xlsx', multiple=False)
label1 = Label("Sube el archivo con la hoja 'Perforación/Perforacion'. Se calculará avance como HASTA - DESDE.")
metros_input = IntText(description="Metros objetivo:", value=10000)
pozos_input = IntText(description="Cantidad de pozos:", value=10)  # (se mantiene, no se usa en el cálculo igual que tu versión)
plazo_input = IntText(description="Plazo (meses):", value=2)
calcular_btn = Button(description="Calcular Sondas", button_style='success')
output = Output()

# ---------------------------
# Utilidades de compatibilidad
# ---------------------------
def _norm(s):
    return str(s).strip().upper() if s is not None else ""

def _pick_sheet(xls, candidates):
    for s in candidates:
        if s in xls.sheet_names:
            return s
    return None

def _resolve_col(df, candidates):
    """Devuelve el nombre real de la primera columna candidata que exista en df (insensible a mayúsculas/acentos)."""
    header_map = {_norm(c): c for c in df.columns}
    for cand in candidates:
        if _norm(cand) in header_map:
            return header_map[_norm(cand)]
    return None

def _to_float_series(s):
    """Convierte textos con miles/puntos/espacios/coma decimal a float de forma segura."""
    s = (s.astype(str)
           .str.replace(u'\xa0','', regex=False)   # NBSP
           .str.replace("'",  '', regex=False)     # miles tipo 1'234
           .str.replace(' ',  '', regex=False)     # miles con espacio
           .str.replace('.',  '', regex=False)     # miles con punto
           .str.replace(',', '.', regex=False))    # decimal
    return pd.to_numeric(s, errors='coerce')

# Función principal
def calcular_sondas(b):
    output.clear_output()
    with output:
        if not upload.value:
            print("⚠️ Primero debes subir un archivo.")
            return

        # Leer archivo subido
        uploaded_file = list(upload.value.values())[0]
        content = BytesIO(uploaded_file['content'])
        try:
            xls = pd.ExcelFile(content)
        except Exception as e:
            print("❌ No se pudo abrir el archivo como Excel:", e)
            return

        # Detectar hoja 'Perforación'/'Perforacion'
        sheet_perfo = _pick_sheet(xls, ["Perforación", "Perforacion"])
        if sheet_perfo is None:
            print("❌ No se encontró la hoja 'Perforación' o 'Perforacion' en el archivo.")
            print("Hojas disponibles:", xls.sheet_names)
            return

        try:
            df = pd.read_excel(xls, sheet_name=sheet_perfo)
        except Exception as e:
            print("❌ Error leyendo la hoja seleccionada:", e)
            return

        # Normalizar columnas
        df.columns = df.columns.str.strip()
        df = df.loc[:, ~df.columns.duplicated()]

        # Resolver columnas requeridas de forma robusta
        col_equipo = _resolve_col(df, ["EQUIPO", "Equipo", "TEAM"])
        col_fecha  = _resolve_col(df, ["FECHA", "Fecha", "FECHA REGISTRO", "DATE"])
        col_desde  = _resolve_col(df, ["DESDE", "Desde", "FROM"])
        col_hasta  = _resolve_col(df, ["HASTA", "Hasta", "TO"])

        if not all([col_equipo, col_fecha, col_desde, col_hasta]):
            print("❌ El archivo debe contener columnas equivalentes a: EQUIPO, FECHA, DESDE, HASTA")
            print("Columnas detectadas:", list(df.columns))
            return

        # Calcular avance y limpiar datos (tolerante a números con coma/puntos/espacios)
        df['_FECHA']  = pd.to_datetime(df[col_fecha], errors='coerce')
        df['_EQUIPO'] = df[col_equipo].astype(str).str.strip()
        df['_DESDE']  = _to_float_series(df[col_desde])
        df['_HASTA']  = _to_float_series(df[col_hasta])

        df['AVANCE'] = df['_HASTA'] - df['_DESDE']
        df = df.dropna(subset=['_FECHA', '_EQUIPO', 'AVANCE'])
        df = df[df['AVANCE'] > 0]  # descartar registros inválidos

        if df.empty:
            print("❌ No hay registros válidos de avance (HASTA - DESDE > 0) tras limpiar los datos.")
            return

        # Calcular rendimiento diario por equipo (igual a tu lógica)
        avance_dia = df.groupby(['_EQUIPO', '_FECHA'])['AVANCE'].sum().reset_index()
        rendimiento_por_equipo = avance_dia.groupby('_EQUIPO')['AVANCE'].mean()

        if rendimiento_por_equipo.empty or rendimiento_por_equipo.mean() <= 0:
            print("❌ No se pudo calcular un rendimiento válido. Revisa los datos de avance.")
            return

        rendimiento_promedio = rendimiento_por_equipo.mean()
        print(f"✅ Rendimiento promedio por sonda: {rendimiento_promedio:.2f} metros por día")

        # Cálculo de sondas necesarias (idéntico a tu versión)
        metros = metros_input.value
        meses  = plazo_input.value
        dias   = meses * 30  # mes ≈ 30 días

        if rendimiento_promedio <= 0:
            print("❌ El rendimiento promedio no puede ser 0.")
            return

        sondas_necesarias = ceil(metros / (rendimiento_promedio * dias))

        print(f"\n📌 Para perforar {metros} metros en {meses} meses (~{dias} días):")
        print(f"🔧 Necesitas al menos {sondas_necesarias} sondas operando continuamente.")

# Vincular botón
calcular_btn.on_click(calcular_sondas)

# Mostrar interfaz
display(VBox([label1, upload, metros_input, pozos_input, plazo_input, calcular_btn, output]))
